# E2 — Text + Pretrained Emoji Embedding (Concatenation Fusion)

This notebook implements the **E2** experiment for the research question:
> *Do emojis carry sentiment information that words miss?*

- **E2 uses `text_without_emoji` + pretrained `emoji_list`** — text branch sees no emojis; emoji branch gets frozen pretrained embeddings from TweetEval.
- Frozen `bert-base-uncased` → mean pooling (768) + **pretrained frozen** emoji embedding (32, mean pooled) → concat (800) → MLP head → 3 classes.
- Class-weighted cross-entropy (weights from train only).
- Run on a **T4 GPU** runtime in Google Colab.


## 1. Setup and Environment Configuration

Import the required libraries, load configuration, set the random seed, and verify GPU availability.


In [ ]:
# Cell: Setup and config
import os, sys, json, random, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# Configuration
CFG = {
    "model_name": "bert-base-uncased",
    "max_length": 128,
    "seed": 42,
    "batch_size": 32,
    "epochs": 5,
    "learning_rate": 0.001,
    "weight_decay": 1e-4,
    "hidden_size": 768,
    "classifier_hidden": 256,
    "dropout": 0.3,
    "n_classes": 3,
    "class_names": ["Bearish", "Neutral", "Bullish"],
    "emoji_dim": 32,
    "max_emojis": 8,
}

# Reproducibility
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

# Output dir
os.makedirs("results/E2", exist_ok=True)
os.makedirs("data/processed/canonical", exist_ok=True)


## 2. Load and Validate Preprocessed Data

Load the final canonical JSONL files and validate the expected row counts and columns.


In [ ]:
# Cell: Load and validate canonical data
# Load canonical JSONL files (use the final approved splits)
train_df = pd.read_json("data/processed/canonical/final_train.jsonl", lines=True)
val_df = pd.read_json("data/processed/canonical/final_validation.jsonl", lines=True)
test_df = pd.read_json("data/processed/canonical/final_test.jsonl", lines=True)

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

# Validate expected counts
expected = {"train": 91121, "validation": 20676, "test": 11966}
assert len(train_df) == expected["train"], "Train count mismatch"
assert len(val_df) == expected["validation"], "Validation count mismatch"
assert len(test_df) == expected["test"], "Test count mismatch"
print("Split counts validated OK")

# Validate required columns
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    assert "text_without_emoji" in df.columns, f"Missing text_without_emoji in {name}"
    assert "emoji_list" in df.columns, f"Missing emoji_list in {name}"
    assert df["text_without_emoji"].notna().all(), f"Null text_without_emoji in {name}"
    assert df["emoji_list"].notna().all(), f"Null emoji_list in {name}"
print("text_without_emoji and emoji_list columns present and non-null in all splits")

# Label validation
for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    assert set(df["label"].unique()) <= {0, 1, 2}, f"Invalid labels in {name}"
print("All labels valid (0/1/2)")

# Preview a row
print("\nSample row:")
sample = train_df.iloc[0]
print(f"  original_text: {sample['original_text'][:80]}...")
print(f"  text_without_emoji: {sample['text_without_emoji'][:80]}...")
print(f"  emoji_list: {sample['emoji_list']}")
print(f"  label: {sample['label']} ({['Bearish','Neutral','Bullish'][sample['label']]})")


## 3. Emoji Vocabulary and Encoding

Build the emoji vocabulary from TRAIN only (no test leakage), and encode emoji lists.


In [ ]:
# Cell: Build emoji vocabulary and encode emoji lists
import json
from collections import Counter

def build_emoji_vocab(emoji_lists, min_freq=1):
    """Build a deterministic emoji vocabulary from training emojis only."""
    counter = Counter()
    for emojis in emoji_lists:
        counter.update(emojis)

    emoji_to_id = {"<UNK>": 0}
    id_to_emoji = {0: "<UNK>"}
    for emoji_char, count in counter.most_common():
        if count >= min_freq:
            eid = len(emoji_to_id)
            emoji_to_id[emoji_char] = eid
            id_to_emoji[eid] = emoji_char

    return {
        "emoji_to_id": emoji_to_id,
        "id_to_emoji": id_to_emoji,
        "vocab_size": len(emoji_to_id),
        "unk_id": 0,
        "counts": dict(counter.most_common(50)),
    }

def encode_emoji_list(emojis, emoji_to_id, max_emojis=8):
    """Encode a list of emojis into padded ids + mask."""
    unk = emoji_to_id.get("<UNK>", 0)
    ids = [emoji_to_id.get(e, unk) for e in emojis[:max_emojis]]
    mask = [1.0] * len(ids)
    pad_len = max_emojis - len(ids)
    if pad_len > 0:
        ids += [0] * pad_len
        mask += [0.0] * pad_len
    return ids, mask

# Build vocab from TRAIN only (no test leakage)
vocab = build_emoji_vocab(train_df["emoji_list"].tolist())
print(f"Emoji vocab size (train-only): {vocab['vocab_size']}")

# Save vocab
os.makedirs("models/emoji_embeddings", exist_ok=True)
with open("models/emoji_embeddings/emoji_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

max_emojis = CFG["max_emojis"]

# Encode emoji lists for all splits
def encode_split(df):
    ids, masks = [], []
    for emojis in df["emoji_list"].tolist():
        i, m = encode_emoji_list(emojis, vocab["emoji_to_id"], max_emojis)
        ids.append(i)
        masks.append(m)
    return np.array(ids), np.array(masks)

e_train_ids, e_train_masks = encode_split(train_df)
e_val_ids, e_val_masks = encode_split(val_df)
e_test_ids, e_test_masks = encode_split(test_df)

print(f"Train emoji ids: {e_train_ids.shape}, masks: {e_train_masks.shape}")
print(f"Val emoji ids: {e_val_ids.shape}, masks: {e_val_masks.shape}")
print(f"Test emoji ids: {e_test_ids.shape}, masks: {e_test_masks.shape}")

# Save vocab for reproducibility
with open("models/emoji_embeddings/emoji_vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
print("Saved emoji vocab -> models/emoji_embeddings/emoji_vocab.json")


## 4. Class Weights and BERT Featurization

Compute class weights from training set only, load frozen BERT, and extract mean-pooled embeddings.


In [ ]:
# Cell: Class weights from TRAIN ONLY (inverse frequency, normalized so mean=1)
from collections import Counter

train_dist = Counter(train_df["label"])
val_dist = Counter(val_df["label"])
test_dist = Counter(test_df["label"])

print("Train distribution:", {["Bearish", "Neutral", "Bullish"][k]: v for k, v in train_dist.items()})
print("Validation distribution:", {["Bearish", "Neutral", "Bullish"][k]: v for k, v in val_dist.items()})
print("Test distribution:", {["Bearish", "Neutral", "Bullish"][k]: v for k, v in test_dist.items()})

total = sum(train_dist.values())
n_classes = CFG["n_classes"]
class_weights = {c: total / (n_classes * train_dist[c]) for c in range(3)}
class_names = ["Bearish", "Neutral", "Bullish"]
print("\nClass weights (train-only):", {class_names[k]: round(v, 4) for k, v in class_weights.items()})

weights_tensor = torch.tensor([class_weights[c] for c in range(3)], dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

# Load frozen BERT tokenizer and model
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])
bert = AutoModel.from_pretrained(CFG["model_name"])

# Freeze encoder
for param in bert.parameters():
    param.requires_grad = False
bert.to(device)
bert.eval()
print("BERT loaded and frozen:", CFG["model_name"])

# Featurization function (cached)
def featurize(texts, tokenizer, bert, max_length, batch_size, device):
    """Extract mean-pooled BERT embeddings for a list of texts."""
    embeds = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length,
                            return_tensors="pt").to(device)
            out = bert(**enc)
            last_hidden = out.last_hidden_state  # (B, L, H)
            mask = enc["attention_mask"].unsqueeze(-1).float()
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = (summed / counts).cpu().numpy()
            embeds.append(pooled)
    return np.concatenate(embeds, axis=0)


CACHE_DIR = "results/E2/embeddings_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def load_or_featurize(df, split_name, tokenizer, bert, cfg):
    path = os.path.join(CACHE_DIR, f"embeds_{split_name}.npy")
    if os.path.exists(path):
        print(f"Loading cached embeddings: {split_name}")
        return np.load(path)
    print(f"Featurizing {split_name}: {len(df)} examples...")
    emb = featurize(df["text_without_emoji"].tolist(), tokenizer, bert,
                    cfg["max_length"], cfg["batch_size"], device)
    np.save(path, emb)
    print(f"Saved embeddings for {split_name} -> {path}")
    return emb

X_train = load_or_featurize(train_df, "train", tokenizer, bert, CFG)
X_val = load_or_featurize(val_df, "validation", tokenizer, bert, CFG)
X_test = load_or_featurize(test_df, "test", tokenizer, bert, CFG)

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print("Embedding shapes:", X_train.shape, X_val.shape, X_test.shape)


## 5. Pretrained Emoji Encoder (from TweetEval)

Load the pretrained emoji embeddings learned from TweetEval emoji prediction task.


In [ ]:
# Cell: Load pretrained emoji encoder from TweetEval
class PretrainedEmojiEncoder(torch.nn.Module):
    """Emoji encoder using pretrained embeddings (frozen for E2)."""
    def __init__(self, pretrained_path, vocab_size, embedding_dim=32, max_emojis=8):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.max_emojis = max_emojis
        
        # Load pretrained embeddings
        pretrained = torch.load(pretrained_path, map_location="cpu")
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=None)
        self.embedding.weight.data = pretrained["emoji_embedding"]
        
        # Freeze embeddings
        for param in self.embedding.parameters():
            param.requires_grad = False

    def forward(self, emoji_ids, emoji_masks):
        """Aggregate emoji embeddings via mean pooling."""
        emb = self.embedding(emoji_ids)  # (B, M, D)
        mask = emoji_masks.unsqueeze(-1).float()  # (B, M, 1)
        summed = (emb * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts  # (B, D)

# Load pretrained emoji encoder
pretrained_path = "models/emoji_embeddings/pretrained_emoji_32d.pt"
if os.path.exists(pretrained_path):
    emoji_encoder = PretrainedEmojiEncoder(
        pretrained_path=pretrained_path,
        vocab_size=vocab["vocab_size"],
        embedding_dim=CFG["emoji_dim"],
        max_emojis=CFG["max_emojis"]
    ).to(device)
    print(f"Loaded pretrained emoji embeddings from {pretrained_path}")
else:
    raise FileNotFoundError(
        f"Pretrained emoji embeddings not found at {pretrained_path}. "
        "Run src/train_e2_pretrain.py first to generate them."
    )

# Encode emoji lists for all splits using the trained vocab
def encode_emoji_list(emojis, emoji_to_id, max_emojis=8):
    unk = emoji_to_id.get("<UNK>", 0)
    ids = [emoji_to_id.get(e, unk) for e in emojis[:max_emojis]]
    mask = [1.0] * len(ids)
    pad_len = max_emojis - len(ids)
    if pad_len > 0:
        ids += [0] * pad_len
        mask += [0.0] * pad_len
    return ids, mask

def encode_split(df):
    ids, masks = [], []
    for emojis in df["emoji_list"].tolist():
        i, m = encode_emoji_list(emojis, vocab["emoji_to_id"], CFG["max_emojis"])
        ids.append(i)
        masks.append(m)
    return np.array(ids), np.array(masks)

e_train_ids, e_train_masks = encode_split(train_df)
e_val_ids, e_val_masks = encode_split(val_df)
e_test_ids, e_test_masks = encode_split(test_df)

print(f"Train emoji ids: {e_train_ids.shape}, masks: {e_train_masks.shape}")
print(f"Val emoji ids: {e_val_ids.shape}, masks: {e_val_masks.shape}")
print(f"Test emoji ids: {e_test_ids.shape}, masks: {e_test_masks.shape}")

# Convert to tensors
e_train_ids_t = torch.tensor(e_train_ids, dtype=torch.long)
e_train_masks_t = torch.tensor(e_train_masks, dtype=torch.float32)
e_val_ids_t = torch.tensor(e_val_ids, dtype=torch.long)
e_val_masks_t = torch.tensor(e_val_masks, dtype=torch.float32)
e_test_ids_t = torch.tensor(e_test_ids, dtype=torch.long)
e_test_masks_t = torch.tensor(e_test_masks, dtype=torch.float32)


## 6. E2 Concat-Fusion Model Definition

Define the concatenation fusion model: frozen BERT (768) + pretrained emoji embedding (32) → concat (800) → MLP head.


In [ ]:
# Cell: E2 Concat-Fusion Model Definition

class E2ConcatFusionModel(torch.nn.Module):
    """E2: Text + pretrained emoji embedding, concatenation fusion."""
    def __init__(self, model_name="bert-base-uncased", text_dim=768, emoji_dim=32,
                 num_labels=3, dropout=0.3, classifier_hidden=256,
                 freeze_encoder=True, pretrained_emoji_path=None):
        super().__init__()
        self.model_name = model_name
        self.text_dim = text_dim
        self.emoji_dim = emoji_dim
        self.fused_dim = text_dim + emoji_dim  # 800

        # Text branch: frozen BERT
        from transformers import AutoConfig, AutoModel
        config = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
        self.encoder = AutoModel.from_pretrained(model_name, config=config)
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # Emoji branch: pretrained frozen embeddings
        self.emoji_encoder = torch.nn.Embedding(32, 32, padding_idx=None)
        pretrained = torch.load(pretrained_emoji_path, map_location="cpu")
        self.emoji_encoder.weight.data = pretrained["emoji_embedding"]
        for param in self.emoji_encoder.parameters():
            param.requires_grad = False

        # Mean pooling for emoji aggregation
        self.max_emojis = 8

        # Classification head
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(text_dim + emoji_dim, classifier_hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(classifier_hidden, num_labels),
        )

    def emoji_forward(self, emoji_ids, emoji_masks):
        """Mean pool emoji embeddings."""
        emb = self.emoji_encoder(emoji_ids)  # (B, M, 32)
        mask = emoji_masks.unsqueeze(-1).float()  # (B, M, 1)
        summed = (emb * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts  # (B, 32)

    def forward(self, input_ids, attention_mask, emoji_ids, emoji_masks):
        # Text branch
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        last_hidden = outputs.last_hidden_state  # (B, L, H)
        mask = attention_mask.unsqueeze(-1).float()  # (B, L, 1)
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        text_rep = summed / counts  # (B, 768)

        # Emoji branch
        emoji_rep = self.emoji_forward(emoji_ids, emoji_masks)  # (B, 32)

        # Fusion: concatenation
        fused = torch.cat([text_rep, emoji_rep], dim=-1)  # (B, 800)
        logits = self.classifier(fused)
        return logits


# Initialize model
model = E2ConcatFusionModel(
    model_name=CFG["model_name"],
    text_dim=768,
    emoji_dim=CFG["emoji_dim"],
    num_labels=CFG["n_classes"],
    dropout=CFG["dropout"],
    classifier_hidden=CFG["classifier_hidden"],
    freeze_encoder=True,
    pretrained_emoji_path="models/emoji_embeddings/pretrained_emoji_32d.pt",
)
model.to(device)

# Verify dimensions
print("E2 Model initialized:")
print(f"  Text embedding dim: 768")
print(f"  Emoji embedding dim: {CFG['emoji_dim']}")
print(f"  Fused dim: 800")
print(f"  Classifier: 800 → 256 → 3")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# Verify text branch uses text_without_emoji (no emojis)
print("\n✓ E2 uses text_without_emoji for text branch (verified by e2_concat_fusion.py)")


## 7. Training Loop with Class-Weighted Loss

Train the head with class-weighted cross-entropy, validate each epoch, save best by validation macro-F1.


In [ ]:
# Cell: Training loop with class-weighted loss

weights_tensor = torch.tensor([class_weights[c] for c in range(3)], dtype=torch.float32).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights_tensor)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"]
)

# Prepare tensors
e_train_ids_t = torch.tensor(e_train_ids, dtype=torch.long)
e_train_masks_t = torch.tensor(e_train_masks, dtype=torch.float32)
e_val_ids_t = torch.tensor(e_val_ids, dtype=torch.long)
e_val_masks_t = torch.tensor(e_val_masks, dtype=torch.float32)
e_test_ids_t = torch.tensor(e_test_ids, dtype=torch.long)
e_test_masks_t = torch.tensor(e_test_masks, dtype=torch.float32)

# Create datasets
train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(e_train_ids, dtype=torch.long),
    torch.tensor(e_train_masks, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
val_ds = TensorDataset(
    torch.tensor(X_val, dtype=torch.float32),
    torch.tensor(e_val_ids, dtype=torch.long),
    torch.tensor(e_val_masks, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.long)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(e_test_ids, dtype=torch.long),
    torch.tensor(e_test_masks, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False)

best_val_f1 = -1.0
best_epoch = -1
patience = 5
no_improve = 0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
start = time.time()

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    total_loss, n_batches = 0.0, 0
    for text_emb, e_ids, e_mask, lab in train_loader:
        text_emb, lab = text_emb.to(device), lab.to(device)
        e_ids, e_mask = e_ids.to(device), e_mask.to(device)
        optimizer.zero_grad()
        # Forward through concatenated representation
        emoji_rep = model.emoji_forward(e_ids, e_mask)
        fused = torch.cat([text_emb, emoji_rep], dim=-1)
        logits = model.classifier(fused)
        loss = criterion(logits, lab)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    train_loss = total_loss / max(n_batches, 1)

    # Validation
    model.eval()
    val_loss, preds, true = 0.0, [], []
    with torch.no_grad():
        for text_emb, e_ids, e_mask, lab in val_loader:
            text_emb, lab = text_emb.to(device), lab.to(device)
            e_ids, e_mask = e_ids.to(device), e_mask.to(device)
            emoji_rep = model.emoji_encoder(e_ids, e_mask)
            fused = torch.cat([text_emb, emoji_rep], dim=-1)
            logits = model.classifier(fused)
            val_loss += criterion(logits, lab).item()
            preds.extend(logits.argmax(dim=-1).cpu().numpy().tolist())
            true.extend(lab.cpu().numpy().tolist())
    val_loss /= max(len(val_loader), 1)
    val_acc = accuracy_score(true, preds)
    val_f1 = f1_score(true, preds, average="macro", zero_division=0)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch}/{CFG['epochs']} | TrainLoss {train_loss:.4f} | "
          f"ValLoss {val_loss:.4f} | ValAcc {val_acc:.4f} | ValMacroF1 {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        no_improve = 0
        torch.save({
            "classifier_state": model.classifier.state_dict(),
            "emoji_vocab_size": model.emoji_encoder.vocab_size,
            "best_val_f1": val_f1,
            "best_epoch": epoch,
            "class_weights": class_weights,
        }, "results/E2/best_model.pt")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

training_time = time.time() - start
print(f"\nTraining complete in {training_time:.1f}s. Best epoch: {best_epoch}, Best val macro-F1: {best_val_f1:.4f}")


## 8. Evaluation on Test Set

Load the best checkpoint, run inference on the test set, compute metrics, and build the confusion matrix.


In [ ]:
# Cell: Evaluate on test set
# Load best checkpoint
ckpt = torch.load("results/E2/best_model.pt", map_location="cpu")
model.classifier.load_state_dict(ckpt["classifier_state"])
model.to(device)
model.eval()
print(f"Loaded best model (epoch {ckpt['best_epoch']}, val macro-F1 {ckpt['best_val_f1']:.4f})")

# Inference on test
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(e_test_ids, dtype=torch.long),
    torch.tensor(e_test_masks, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False)
all_probs = []
with torch.no_grad():
    for text_emb, e_ids, e_mask, _ in test_loader:
        text_emb = text_emb.to(device)
        e_ids = e_ids.to(device)
        e_mask = e_mask.to(device)
        emoji_rep = model.emoji_forward(e_ids, e_mask)
        fused = torch.cat([text_emb, emoji_rep], dim=-1)
        logits = model.classifier(fused)
        all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
probs = np.concatenate(all_probs)
y_pred = probs.argmax(axis=1)

# Metrics
acc = accuracy_score(y_test, y_pred)
macro_p = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_r = recall_score(y_test, y_pred, average="macro", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"Accuracy: {acc:.4f} | MacroP: {macro_p:.4f} | MacroR: {macro_r:.4f} | MacroF1: {macro_f1:.4f}")

# Classification report
class_names = ["Bearish", "Neutral", "Bullish"]
report = classification_report(y_test, y_pred, labels=[0, 1, 2],
                               target_names=class_names, digits=4, zero_division=0)
print("\nClassification report:\n", report)

# Confusion matrices
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
class_names = ["Bearish", "Neutral", "Bullish"]
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=class_names, yticklabels=class_names)
axes[0].set_title("Confusion Matrix (counts)")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=axes[1],
            xticklabels=class_names, yticklabels=class_names)
axes[1].set_title("Confusion Matrix (normalized)")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.savefig("results/E2/confusion_matrix.png", dpi=150)
plt.show()
print("Saved confusion matrix -> results/E2/confusion_matrix.png")


## 9. Error Analysis and Visualization

Identify misclassified examples (with removed emoji info), plot training curves, and save the error table.


In [ ]:
# Cell: Error analysis + training curves
# Attach predictions to test dataframe
test_out = test_df.copy()
test_out["predicted_label"] = y_pred
test_out["pred_conf_bearish"] = probs[:, 0]
test_out["pred_conf_neutral"] = probs[:, 1]
test_out["pred_conf_bullish"] = probs[:, 2]
test_out["correct"] = (y_test == y_pred)
test_out.to_csv("results/E2/predictions.csv", index=False)

# Error analysis: misclassified examples
errors = test_out[~test_out["correct"]].copy()
print(f"Misclassified: {len(errors)} / {len(test_out)}")
errors.to_csv("results/E2/error_analysis.csv", index=False)
print("Saved error analysis -> results/E2/error_analysis.csv")

# Show a few error examples with emojis that were removed
print("\nSample error cases (with removed emoji info):")
class_names = ["Bearish", "Neutral", "Bullish"]
for _, row in errors.sort_values("pred_conf_bullish", ascending=False).head(10).iterrows():
    print(f"  [{class_names[row['label']]}->{class_names[row['predicted_label']]}] {row['text_without_emoji'][:50]!r} | emojis={row['emoji_list']}")

# Training curves
epochs_arr = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(epochs_arr, history["train_loss"], marker="o", label="Train Loss")
axes[0].plot(epochs_arr, history["val_loss"], marker="o", label="Val Loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(epochs_arr, history["val_acc"], marker="o", color="green")
axes[1].set_title("Validation Accuracy"); axes[1].set_xlabel("Epoch")
axes[2].plot(epochs_arr, history["val_f1"], marker="o", color="purple")
axes[2].set_title("Validation Macro-F1"); axes[2].set_xlabel("Epoch")
plt.tight_layout()
plt.savefig("results/E2/training_history.png", dpi=150)
plt.show()
print("Saved training curves -> results/E2/training_history.png")

# Confidence distribution
plt.figure(figsize=(8, 5))
sns.histplot(probs.max(axis=1), bins=50)
plt.title("Prediction Confidence Distribution")
plt.xlabel("Max softmax probability")
plt.savefig("results/E2/confidence_distribution.png", dpi=150)
plt.show()


## 10. Results Summary and Comparison

Compile all metrics to `metrics.json`, generate `E2_report.md`, and save a config snapshot for reproducibility.
